# M.A.C.H. academic tool router — QLoRA pilot

This notebook trains the tool-selection and argument-extraction behavior only. The original documents and databases remain the source of academic facts.

Before running: in Kaggle Notebook settings enable **GPU** and **Internet**. This pilot uses the curated-only dataset (202 train / 69 held-out examples). The examples are authored seed data, not domain-admin reviewed.

In [ ]:
!nvidia-smi
%pip install -q --upgrade "transformers>=5.10.1" "trl" "peft" "datasets>=4.7.0" "bitsandbytes" "accelerate"

In [ ]:
from pathlib import Path
import zipfile

archives = list(Path("/kaggle/input").rglob("campus_router_training.zip"))
if not archives:
    raise FileNotFoundError("Attach the Kaggle dataset containing campus_router_training.zip")
workdir = Path("/kaggle/working/router_training")
workdir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archives[0]) as archive:
    archive.extractall(workdir)
print("Training files:", [p.name for p in workdir.iterdir()])
print("GPU ready; input:", archives[0])

In [ ]:
%cd /kaggle/working/router_training
!python validate_routing_data.py campus_router_curated_v1.jsonl
!python train_router_qlora.py --dataset campus_router_curated_v1.jsonl --model-id google/gemma-4-12B-it --output-dir /kaggle/working/router_output --epochs 2

In [ ]:
from pathlib import Path
import json, shutil

manifest = Path("/kaggle/working/router_output/training_manifest.json")
if manifest.exists():
    print(json.dumps(json.loads(manifest.read_text()), indent=2))
    archive = shutil.make_archive("/kaggle/working/router_output_bundle", "zip", root_dir="/kaggle/working/router_output")
    print("Download artifact:", archive)
else:
    raise FileNotFoundError("Training did not finish; no manifest was produced")

## After the run

Download the `router_output` directory from the notebook output. Review base-vs-tuned exact tool and argument accuracy route by route. Do not switch the live application based on training loss alone. The adapter is not yet integrated with the app’s Ollama endpoint; a separate serving/export compatibility step is required.